# Formal Analysis: Stochastic Process Decomposition & Root Node Influence

This notebook provides a rigorous analysis of the network epistemology simulation as a Markov Chain,
focusing on two key aspects:

1. **Stochastic Process Decomposition**: Breaking down the simulation into its constituent stochastic components
2. **Root Node Influence Analysis**: Understanding how root nodes (sources) determine long-run outcomes

## Theoretical Framework

The simulation is fundamentally a **Markov Chain** where:
- **State Space**: Collection of all agents' belief parameters $S_t = \{(\alpha_i^{(0)}, \beta_i^{(0)}, \alpha_i^{(1)}, \beta_i^{(1)})\}_{i=1}^N$
- **Transitions**: Determined by Bayesian updates from Binomial experiment outcomes
- **Absorbing States**: Consensus states where all agents have converged

# Setup

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# RECOMMENDED GOOGLE COLAB RUNTIME
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("RECOMMENDED COLAB RUNTIME SETTINGS")
print("=" * 70)
print("""
Runtime Type: CPU (NOT GPU/TPU)
   - This notebook uses multiprocessing (CPU parallelization)
   - GPU/TPU won't help and wastes resources

Hardware Accelerator: None
   - Go to: Runtime -> Change runtime type -> Hardware accelerator: None

RAM:
   - Standard (12GB): OK for small networks (n < 200)
   - High-RAM (25GB+): RECOMMENDED for larger networks

Session Duration:
   - Free Colab: ~90 min timeout, may disconnect
   - Colab Pro: Up to 24h runtime

TIP: Run in background with Colab Pro for long simulations!
""")
print("=" * 70)

In [ ]:
# Clone the repository (ai-agents-branch has the latest code)
!git clone -b ai-agents-branch https://github.com/IgnacioOQ/e_network_inequality

In [ ]:
# Install required packages
!pip install dill tqdm networkx pandas numpy scipy matplotlib seaborn

In [ ]:
# Change to repository directory and install the package
%cd e_network_inequality
!pip install -e .

In [ ]:
# Add src to path and import modules
import sys
import os
sys.path.insert(0, os.path.abspath('src'))

# Core imports
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
from multiprocessing import Pool, cpu_count
from functools import partial
from scipy import stats

# Import from net_epistemology package
from net_epistemology.utils.imports import *
from net_epistemology.core.vectorized_model import VectorizedModel
from net_epistemology.utils.network_generation import *

# Import tqdm AFTER the wildcard imports to avoid overwriting
from tqdm.auto import tqdm

print("All imports successful!")

---
# Part 1: Stochastic Process Decomposition

This section decomposes the simulation into its constituent stochastic components to understand
how outcomes are determined.

## The Three Components of Outcome Determination

1. **Stochastic Matrices** - Derived from the sampling and update mechanism (Binomial experiments, Bayesian updating)
2. **Network Structure** - Determines information flow (who observes whom)
3. **Initial Distribution** - Random priors (starting beliefs)

We analyze each component using a Markov chain framework.

## 1.1 Deriving the Expected Update Equations

The key insight is that the update mechanism has a **deterministic expectation** overlaid with **stochastic noise**.

### The Update Rule

At each step, for agent $i$ testing theory $k$:
$$\alpha_i^{(k)}(t+1) = \alpha_i^{(k)}(t) + \sum_{j \in \mathcal{N}(i) \cup \{i\}} S_j^{(k)}(t)$$

where $S_j^{(k)}(t) \sim \text{Binomial}(n, p_k)$ with:
- $p_1 = 0.5 + u$ (truth)
- $p_0 = 0.5 - u$ (false)

### Expected Evidence Flow

Taking expectations over the Binomial sampling:
$$\mathbb{E}[S_j^{(k)}] = n \cdot p_k \cdot \mathbf{1}[\text{agent } j \text{ tests theory } k]$$

The network aggregation matrix $A^T$ (transposed adjacency) determines how evidence flows.

In [ ]:
# ============================================================================
# 1.1 EXPECTED UPDATE EQUATIONS AND TRANSITION MATRICES
# ============================================================================

def compute_listening_matrix(G):
    """
    Compute the row-stochastic listening matrix W.
    
    W[i,j] = weight agent i places on agent j's evidence.
    - If i is a root (no predecessors): W[i,i] = 1 (listens only to self)
    - Otherwise: W[i,j] = 1/|Pred(i)| for j in Pred(i)
    
    Returns:
        W: (n, n) row-stochastic matrix
        nodes: list of node IDs in order
    """
    nodes = list(G.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    W = np.zeros((n, n))
    
    for u in nodes:
        u_idx = node_to_idx[u]
        influencers = list(G.predecessors(u))
        
        if len(influencers) == 0:
            # Root node - listens only to itself
            W[u_idx, u_idx] = 1.0
        else:
            # Equal weight on all predecessors
            weight = 1.0 / len(influencers)
            for inf in influencers:
                v_idx = node_to_idx[inf]
                W[u_idx, v_idx] = weight
    
    return W, nodes


def compute_evidence_aggregation_matrix(G):
    """
    Compute the evidence aggregation matrix E.
    
    E[i,j] = 1 if agent i observes agent j's experiments (including self).
    This is the adjacency matrix transposed plus identity.
    
    The expected evidence for agent i = E[i,:] @ evidence_vector
    """
    nodes = list(G.nodes())
    n = len(nodes)
    
    # Adjacency matrix: A[i,j] = 1 if edge i -> j exists
    A = nx.to_numpy_array(G, nodelist=nodes)
    
    # Evidence aggregation: sum over predecessors + self
    # If A[i,j] = 1 means i -> j, then j observes i
    # So agent j aggregates from all i where A[i,j] = 1, i.e., column j of A
    # Plus self-observation (identity)
    E = A.T + np.eye(n)
    
    return E, nodes


def compute_expected_update_matrix(G, n_experiments, uncertainty):
    """
    Compute the expected evidence increment per step.
    
    For theory k with probability p_k:
    - Expected successes from agent j if j tests theory k: n * p_k
    - Expected failures: n * (1 - p_k)
    
    This returns the matrix that maps "who tests what" to "expected alpha increments".
    """
    E, nodes = compute_evidence_aggregation_matrix(G)
    n = len(nodes)
    
    p_true = 0.5 + uncertainty  # Theory 1
    p_false = 0.5 - uncertainty  # Theory 0
    
    # Expected successes per experiment for each theory
    exp_success_true = n_experiments * p_true
    exp_success_false = n_experiments * p_false
    
    return {
        'E': E,
        'nodes': nodes,
        'exp_success_true': exp_success_true,
        'exp_success_false': exp_success_false,
        'exp_failure_true': n_experiments * (1 - p_true),
        'exp_failure_false': n_experiments * (1 - p_false),
    }

In [ ]:
# Create test networks for analysis
def create_test_networks():
    """
    Create a variety of test networks for analysis.
    Focus on networks WITH root nodes for root influence analysis.
    """
    networks = {}
    
    print("=" * 70)
    print("CREATING TEST NETWORKS")
    print("=" * 70)
    
    # 1. Barabasi-Albert (scale-free with hubs)
    print("\n[1] Creating Barabasi-Albert network...")
    G_ba = nx.barabasi_albert_graph(100, 3, seed=42)
    G_ba = nx.DiGraph(G_ba)  # Convert to directed
    root_count_ba = sum(1 for n in G_ba.nodes() if G_ba.in_degree(n) == 0)
    networks['barabasi_albert'] = {
        'graph': G_ba,
        'has_roots': root_count_ba > 0,
        'description': f'Barabasi-Albert (100 nodes, {root_count_ba} roots)'
    }
    print(f"     {root_count_ba} root nodes detected")
    
    # 2. Directed Tree (perfect hierarchy)
    print("\n[2] Creating Directed Tree network...")
    G_tree = nx.balanced_tree(3, 4, create_using=nx.DiGraph())
    # Reverse edges so root has influence (children listen to parents)
    G_tree = G_tree.reverse()
    networks['tree'] = {
        'graph': G_tree,
        'has_roots': True,
        'description': f'Balanced Tree (3-ary depth 4, 1 root with 100% influence)'
    }
    print("     1 root node (perfect hierarchy)")
    
    # 3. Watts-Strogatz (small-world)
    print("\n[3] Creating Watts-Strogatz network...")
    G_ws = nx.watts_strogatz_graph(100, 4, 0.3, seed=42)
    G_ws = nx.DiGraph(G_ws)
    root_count_ws = sum(1 for n in G_ws.nodes() if G_ws.in_degree(n) == 0)
    networks['watts_strogatz'] = {
        'graph': G_ws,
        'has_roots': root_count_ws > 0,
        'description': f'Watts-Strogatz (100 nodes, {root_count_ws} roots)'
    }
    print(f"     {root_count_ws} root nodes detected")
    
    # 4. Load empirical network if available
    try:
        network_path = 'data/empirical_networks/pud_final.json'
        with open(network_path, 'r') as f:
            network_data = json.load(f)
        if 'links' in network_data:
            network_data['edges'] = network_data.pop('links')
        G_emp = nx.node_link_graph(network_data)
        root_count = sum(1 for n in G_emp.nodes() if G_emp.in_degree(n) == 0)
        networks['empirical_pud'] = {
            'graph': G_emp,
            'has_roots': root_count > 0,
            'description': f'Empirical PUD ({len(G_emp.nodes())} nodes, {root_count} roots)'
        }
        print(f"\n[4] Loaded empirical network: {len(G_emp.nodes())} nodes, {root_count} roots")
    except FileNotFoundError:
        print("\n[4] Empirical network not found, skipping...")
    
    return networks


# Create networks
networks = create_test_networks()

# Print summary
print("\n" + "=" * 70)
print("TEST NETWORKS SUMMARY")
print("=" * 70)
print("\n{:<25} {:>8} {:>8} {:>10}".format("Network", "Nodes", "Edges", "Roots"))
print("-" * 55)
for name, info in networks.items():
    G = info['graph']
    roots = sum(1 for n in G.nodes() if G.in_degree(n) == 0)
    print(f"{name:<25} {len(G.nodes()):>8} {len(G.edges()):>8} {roots:>10}")

In [ ]:
# Demonstrate matrix computations on a test network
print("=" * 70)
print("EXPECTED UPDATE EQUATIONS - DEMONSTRATION")
print("=" * 70)

# Use Barabasi-Albert network
G_test = networks['barabasi_albert']['graph']
n_test = len(G_test.nodes())

W, nodes = compute_listening_matrix(G_test)
E, _ = compute_evidence_aggregation_matrix(G_test)
update_info = compute_expected_update_matrix(G_test, n_experiments=10, uncertainty=0.001)

print(f"\nNetwork: {n_test} nodes, {len(G_test.edges())} edges")
print(f"\nListening Matrix W (row-stochastic):")
print(f"  Shape: {W.shape}")
print(f"  Row sums (should be 1): min={W.sum(axis=1).min():.4f}, max={W.sum(axis=1).max():.4f}")
print(f"  Sparsity: {(W == 0).sum() / W.size:.2%} zeros")

print(f"\nEvidence Aggregation Matrix E:")
print(f"  Shape: {E.shape}")
print(f"  Max in-degree + 1: {E.sum(axis=1).max():.0f}")
print(f"  Min in-degree + 1: {E.sum(axis=1).min():.0f}")

print(f"\nExpected evidence per step (n_exp=10, u=0.001):")
print(f"  Theory 1 (truth): {update_info['exp_success_true']:.3f} successes, {update_info['exp_failure_true']:.3f} failures")
print(f"  Theory 0 (false): {update_info['exp_success_false']:.3f} successes, {update_info['exp_failure_false']:.3f} failures")
print(f"  Advantage per step: {update_info['exp_success_true'] - update_info['exp_success_false']:.4f} more successes for truth")

## 1.2 Variance Analysis and Confidence Intervals

The Binomial sampling introduces variance that affects outcome predictability.

### Variance of a Single Update

For agent $i$ aggregating from $d_i$ sources (predecessors + self):
$$\text{Var}(S_i^{(k)}) = \sum_{j \in \mathcal{N}(i) \cup \{i\}} n \cdot p_k (1-p_k) = d_i \cdot n \cdot p_k(1-p_k)$$

### Variance Accumulation Over Time

After $T$ steps, the total variance in $\alpha_i^{(k)}$ scales as $O(T \cdot d_i)$.

However, the **credence** $c = \alpha/(\alpha+\beta)$ has variance that **decreases** as $\alpha + \beta$ grows:
$$\text{Var}(c) \approx \frac{\alpha \beta}{(\alpha+\beta)^2(\alpha+\beta+1)} \to 0 \text{ as } T \to \infty$$

This explains why longer simulations give more consistent outcomes.

In [ ]:
# ============================================================================
# 1.2 VARIANCE ANALYSIS AND CONFIDENCE INTERVALS
# ============================================================================

def compute_update_variance(G, n_experiments, uncertainty):
    """
    Compute the variance of evidence updates for each agent.
    
    Variance of Binomial(n, p) = n * p * (1-p)
    Total variance for agent i = sum over all observed agents
    """
    E, nodes = compute_evidence_aggregation_matrix(G)
    n = len(nodes)
    
    p_true = 0.5 + uncertainty
    p_false = 0.5 - uncertainty
    
    # Variance per experiment
    var_per_exp_true = n_experiments * p_true * (1 - p_true)
    var_per_exp_false = n_experiments * p_false * (1 - p_false)
    
    # Number of sources each agent observes
    n_sources = E.sum(axis=1)  # Row sums
    
    # Total variance per step for each agent
    var_per_step_true = n_sources * var_per_exp_true
    var_per_step_false = n_sources * var_per_exp_false
    
    return {
        'n_sources': n_sources,
        'var_per_step_true': var_per_step_true,
        'var_per_step_false': var_per_step_false,
        'var_per_exp_true': var_per_exp_true,
        'var_per_exp_false': var_per_exp_false,
    }


def run_variance_analysis(G, network_name, n_experiments=10, uncertainty=0.001, 
                          n_trials=50, n_steps=10000):
    """
    Run multiple simulations to empirically measure variance in outcomes.
    """
    print(f"\n{'='*60}")
    print(f"Variance Analysis: {network_name}")
    print(f"Running {n_trials} trials with {n_steps} steps each...")
    print(f"{'='*60}")
    
    nodes = list(G.nodes())
    n = len(nodes)
    
    # Theoretical variance
    var_info = compute_update_variance(G, n_experiments, uncertainty)
    
    # Collect outcomes across trials
    final_credences = []
    final_beliefs = []
    proportions = []
    
    for trial in tqdm(range(n_trials), desc="Trials"):
        model = VectorizedModel(
            network=G,
            n_experiments=n_experiments,
            uncertainty=uncertainty,
            agent_type="beta",
            tstep_stopping=True,
        )
        model.run_simulation(number_of_steps=n_steps, show_bar=False)
        
        final_credences.append(model.credences.copy())
        beliefs = model.credences[:, 1] > model.credences[:, 0]
        final_beliefs.append(beliefs)
        proportions.append(np.mean(beliefs))
    
    final_credences = np.array(final_credences)  # (n_trials, n_agents, 2)
    final_beliefs = np.array(final_beliefs)  # (n_trials, n_agents)
    proportions = np.array(proportions)
    
    # Compute statistics
    # Per-agent credence variance across trials
    credence_var = final_credences.var(axis=0)  # (n_agents, 2)
    credence_std = final_credences.std(axis=0)
    
    # Per-agent belief consistency (fraction of trials with same outcome)
    belief_consistency = np.maximum(final_beliefs.mean(axis=0), 1 - final_beliefs.mean(axis=0))
    
    # Overall proportion variance
    prop_mean = proportions.mean()
    prop_std = proportions.std()
    prop_ci_95 = (np.percentile(proportions, 2.5), np.percentile(proportions, 97.5))
    
    results = {
        'theoretical_var_per_step': var_info,
        'empirical_credence_var': credence_var,
        'empirical_credence_std': credence_std,
        'belief_consistency': belief_consistency,
        'proportion_mean': prop_mean,
        'proportion_std': prop_std,
        'proportion_ci_95': prop_ci_95,
        'final_credences': final_credences,
        'final_beliefs': final_beliefs,
        'proportions': proportions,
    }
    
    print(f"\nResults:")
    print(f"  Proportion believing truth: {prop_mean:.4f} +/- {prop_std:.4f}")
    print(f"  95% CI: [{prop_ci_95[0]:.4f}, {prop_ci_95[1]:.4f}]")
    print(f"  Agent belief consistency: {belief_consistency.mean():.4f} (1.0 = always same outcome)")
    print(f"  Credence std (Theory 1): mean={credence_std[:, 1].mean():.4f}, max={credence_std[:, 1].max():.4f}")
    
    return results

In [ ]:
# Run variance analysis on test networks
print("\n" + "=" * 70)
print("VARIANCE ANALYSIS")
print("=" * 70)

# Store results for comparison
variance_results = {}

# Run on networks with roots
for net_name in ['barabasi_albert', 'tree']:
    if net_name in networks:
        G = networks[net_name]['graph']
        variance_results[net_name] = run_variance_analysis(
            G, net_name, 
            n_experiments=10, 
            uncertainty=0.001,
            n_trials=30,
            n_steps=50000
        )

In [ ]:
# Visualize variance analysis results
n_plots = len(variance_results)
fig, axes = plt.subplots(2, n_plots, figsize=(7*n_plots, 10))

if n_plots == 1:
    axes = axes.reshape(-1, 1)

for idx, (net_name, results) in enumerate(variance_results.items()):
    col = idx
    
    # Plot 1: Distribution of final proportions
    ax = axes[0, col]
    ax.hist(results['proportions'], bins=15, edgecolor='black', alpha=0.7)
    ax.axvline(results['proportion_mean'], color='red', linestyle='--', 
               label=f"Mean: {results['proportion_mean']:.3f}")
    ax.axvline(results['proportion_ci_95'][0], color='orange', linestyle=':', 
               label=f"95% CI: [{results['proportion_ci_95'][0]:.3f}, {results['proportion_ci_95'][1]:.3f}]")
    ax.axvline(results['proportion_ci_95'][1], color='orange', linestyle=':')
    ax.set_xlabel('Proportion Believing Truth')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{net_name}\nDistribution of Outcomes ({len(results["proportions"])} trials)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Agent-level belief consistency
    ax = axes[1, col]
    consistency = results['belief_consistency']
    ax.hist(consistency, bins=20, edgecolor='black', alpha=0.7)
    ax.axvline(0.5, color='gray', linestyle='--', label='Random (0.5)')
    ax.axvline(consistency.mean(), color='red', linestyle='--', 
               label=f'Mean: {consistency.mean():.3f}')
    ax.set_xlabel('Belief Consistency (max of P(truth), P(false))')
    ax.set_ylabel('Number of Agents')
    ax.set_title(f'Per-Agent Outcome Consistency\n(1.0 = same outcome every trial)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('variance_analysis.png', dpi=150, bbox_inches='tight')
print("\nPlot saved to 'variance_analysis.png'")
plt.show()

# Summary comparison
print("\n" + "=" * 70)
print("VARIANCE COMPARISON SUMMARY")
print("=" * 70)
print(f"\n{'Network':<25} {'Prop Mean':>12} {'Prop Std':>12} {'Consistency':>12}")
print("-" * 65)
for net_name, results in variance_results.items():
    print(f"{net_name:<25} {results['proportion_mean']:>12.4f} {results['proportion_std']:>12.4f} {results['belief_consistency'].mean():>12.4f}")

## 1.3 Separating Initial Conditions from Network Structure

A key question: How much of the outcome is determined by:
1. **Initial priors** (random at t=0)
2. **Network structure** (fixed, determines information flow)
3. **Stochastic sampling** (ongoing randomness)

### Experimental Design

To separate these effects, we:
1. **Fix initial conditions** - Run multiple simulations with the SAME initial priors but different sampling randomness
2. **Vary initial conditions** - Run multiple simulations with different initial priors but same network

This allows us to decompose: $\text{Var}(\text{outcome}) = \text{Var}_{\text{init}} + \text{Var}_{\text{sampling}}$

In [ ]:
# ============================================================================
# 1.3 SEPARATING INITIAL CONDITIONS FROM NETWORK STRUCTURE
# ============================================================================

def create_model_with_fixed_init(G, n_experiments, uncertainty, initial_alphas_betas):
    """
    Create a VectorizedModel with fixed initial conditions.
    
    This allows us to isolate the effect of sampling randomness from initial conditions.
    """
    model = VectorizedModel(
        network=G,
        n_experiments=n_experiments,
        uncertainty=uncertainty,
        agent_type="beta",
        tstep_stopping=True,
    )
    # Override the random initialization with fixed values
    model.alphas_betas = initial_alphas_betas.copy()
    # Recompute credences from the fixed alphas_betas
    a = model.alphas_betas[:, :, 0]
    b = model.alphas_betas[:, :, 1]
    model.credences = a / (a + b)
    return model


def run_fixed_init_analysis(G, network_name, n_experiments=10, uncertainty=0.001,
                            n_init_conditions=10, n_sampling_trials=20, n_steps=50000):
    """
    Analyze variance decomposition between initial conditions and sampling.
    
    For each of n_init_conditions different starting points:
      - Run n_sampling_trials simulations with different random seeds
      - Measure within-init variance (due to sampling)
    
    Then measure between-init variance (due to different starting points).
    """
    print(f"\n{'='*60}")
    print(f"Initial Conditions Analysis: {network_name}")
    print(f"Testing {n_init_conditions} initial conditions x {n_sampling_trials} trials")
    print(f"{'='*60}")
    
    nodes = list(G.nodes())
    n = len(nodes)
    
    # Store results
    all_proportions = []  # (n_init, n_trials)
    init_means = []
    within_vars = []
    
    for init_idx in tqdm(range(n_init_conditions), desc="Initial conditions"):
        # Create a random initial condition
        np.random.seed(init_idx * 1000)  # Reproducible but different
        initial_ab = np.zeros((n, 2, 2))
        for i in range(n):
            initial_ab[i, 0] = np.random.uniform(0, 4, size=2)
            initial_ab[i, 1] = np.random.uniform(0, 4, size=2)
        
        # Run multiple trials with this fixed init
        trial_proportions = []
        for trial in range(n_sampling_trials):
            model = create_model_with_fixed_init(G, n_experiments, uncertainty, initial_ab)
            model.run_simulation(number_of_steps=n_steps, show_bar=False)
            beliefs = model.credences[:, 1] > model.credences[:, 0]
            trial_proportions.append(np.mean(beliefs))
        
        trial_proportions = np.array(trial_proportions)
        all_proportions.append(trial_proportions)
        init_means.append(trial_proportions.mean())
        within_vars.append(trial_proportions.var())
    
    all_proportions = np.array(all_proportions)  # (n_init, n_trials)
    init_means = np.array(init_means)
    within_vars = np.array(within_vars)
    
    # Compute variance decomposition
    # Total variance = Between-init variance + Within-init variance (sampling)
    total_var = all_proportions.flatten().var()
    between_var = init_means.var()  # Variance of means across init conditions
    within_var_mean = within_vars.mean()  # Average within-init variance
    
    # Fraction explained by each
    frac_init = between_var / total_var if total_var > 0 else 0
    frac_sampling = within_var_mean / total_var if total_var > 0 else 0
    
    results = {
        'all_proportions': all_proportions,
        'init_means': init_means,
        'within_vars': within_vars,
        'total_var': total_var,
        'between_var': between_var,
        'within_var_mean': within_var_mean,
        'frac_init': frac_init,
        'frac_sampling': frac_sampling,
    }
    
    print(f"\nVariance Decomposition:")
    print(f"  Total variance:           {total_var:.6f}")
    print(f"  Between-init variance:    {between_var:.6f} ({frac_init*100:.1f}% of total)")
    print(f"  Within-init variance:     {within_var_mean:.6f} ({frac_sampling*100:.1f}% of total)")
    print(f"\n  --> Initial conditions explain {frac_init*100:.1f}% of outcome variance")
    print(f"  --> Sampling randomness explains {frac_sampling*100:.1f}% of outcome variance")
    
    return results

In [ ]:
# Run the analysis
print("\n" + "=" * 70)
print("INITIAL CONDITIONS vs SAMPLING VARIANCE DECOMPOSITION")
print("=" * 70)

init_analysis_results = {}

for net_name in ['barabasi_albert', 'tree']:
    if net_name in networks:
        G = networks[net_name]['graph']
        init_analysis_results[net_name] = run_fixed_init_analysis(
            G, net_name,
            n_experiments=10,
            uncertainty=0.001,
            n_init_conditions=10,
            n_sampling_trials=15,
            n_steps=50000
        )

In [ ]:
# Visualize the variance decomposition
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Variance decomposition bar chart
ax = axes[0]
x = np.arange(len(init_analysis_results))
width = 0.35
init_fracs = [r['frac_init'] for r in init_analysis_results.values()]
samp_fracs = [r['frac_sampling'] for r in init_analysis_results.values()]

bars1 = ax.bar(x - width/2, init_fracs, width, label='Initial Conditions', color='steelblue')
bars2 = ax.bar(x + width/2, samp_fracs, width, label='Sampling Randomness', color='coral')

ax.set_ylabel('Fraction of Total Variance')
ax.set_title('Variance Decomposition\n(What determines outcomes?)')
ax.set_xticks(x)
ax.set_xticklabels(init_analysis_results.keys())
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars1, init_fracs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val*100:.0f}%', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, samp_fracs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val*100:.0f}%', ha='center', va='bottom', fontsize=9)

# Plot 2: Distribution of outcomes by initial condition (first network)
ax = axes[1]
first_net = list(init_analysis_results.keys())[0]
results = init_analysis_results[first_net]
for init_idx, trial_props in enumerate(results['all_proportions']):
    ax.scatter([init_idx] * len(trial_props), trial_props, alpha=0.5, s=20)
ax.plot(range(len(results['init_means'])), results['init_means'], 'ko-', 
        label='Mean per init', markersize=8)
ax.set_xlabel('Initial Condition Index')
ax.set_ylabel('Proportion Believing Truth')
ax.set_title(f'{first_net}\nOutcomes by Initial Condition')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Within-init variance distribution
ax = axes[2]
for net_name, results in init_analysis_results.items():
    ax.hist(results['within_vars'], bins=10, alpha=0.5, label=net_name, edgecolor='black')
ax.set_xlabel('Within-Init Variance')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Sampling Variance\n(per initial condition)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('init_conditions_analysis.png', dpi=150, bbox_inches='tight')
print("\nPlot saved to 'init_conditions_analysis.png'")
plt.show()

## 1.4 Spectral Analysis of the Listening Matrix

The **spectral properties** of the listening matrix $W$ reveal important dynamics:

### The Spectral Gap

The **spectral gap** $\gamma = 1 - |\lambda_2|$ (where $\lambda_2$ is the second-largest eigenvalue) controls:
- How fast the system "forgets" initial conditions
- The mixing time of the Markov chain
- The rate of convergence to steady-state influence distribution

### Interpretation

- **Large spectral gap**: Fast convergence, outcomes determined quickly
- **Small spectral gap**: Slow convergence, initial conditions persist longer

In [ ]:
# ============================================================================
# 1.4 SPECTRAL ANALYSIS OF THE LISTENING MATRIX
# ============================================================================

def analyze_spectral_properties(G):
    """
    Analyze the spectral properties of the listening matrix W.
    
    Returns eigenvalues, eigenvectors, spectral gap, and mixing time estimate.
    """
    W, nodes = compute_listening_matrix(G)
    n = len(nodes)
    
    # Compute eigenvalues of W
    eigenvalues = np.linalg.eigvals(W)
    
    # Sort by magnitude (descending)
    eigenvalues_sorted = sorted(eigenvalues, key=lambda x: abs(x), reverse=True)
    
    # The largest eigenvalue should be 1 (row-stochastic)
    lambda_1 = abs(eigenvalues_sorted[0])
    
    # Second largest eigenvalue (may be complex)
    lambda_2 = abs(eigenvalues_sorted[1]) if len(eigenvalues_sorted) > 1 else 0
    
    # Spectral gap
    spectral_gap = 1 - lambda_2
    
    # Mixing time estimate: t_mix ~ 1 / spectral_gap
    mixing_time_est = 1 / spectral_gap if spectral_gap > 0 else np.inf
    
    # Stationary distribution (left eigenvector for eigenvalue 1)
    eigenvalues_full, eigenvectors_full = np.linalg.eig(W.T)
    idx = np.argmin(np.abs(eigenvalues_full - 1.0))
    stationary = np.real(eigenvectors_full[:, idx])
    stationary = np.abs(stationary) / np.sum(np.abs(stationary))
    
    # Concentration of influence
    stationary_sorted = np.sort(stationary)[::-1]
    cumsum = np.cumsum(stationary_sorted)
    n_50 = np.searchsorted(cumsum, 0.5) + 1
    n_90 = np.searchsorted(cumsum, 0.9) + 1
    
    return {
        'W': W,
        'nodes': nodes,
        'eigenvalues': eigenvalues_sorted,
        'lambda_1': lambda_1,
        'lambda_2': lambda_2,
        'spectral_gap': spectral_gap,
        'mixing_time_est': mixing_time_est,
        'stationary_distribution': stationary,
        'n_50_influence': n_50,
        'n_90_influence': n_90,
    }


def run_spectral_analysis(networks_dict):
    """
    Run spectral analysis on all networks and compare.
    """
    print("\n" + "=" * 70)
    print("SPECTRAL ANALYSIS OF LISTENING MATRICES")
    print("=" * 70)
    
    results = {}
    
    for net_name, net_info in networks_dict.items():
        G = net_info['graph']
        spec = analyze_spectral_properties(G)
        results[net_name] = spec
        
        n = len(spec['nodes'])
        print(f"\n{net_name}:")
        print(f"  Nodes: {n}")
        print(f"  Lambda_1 (should be 1): {spec['lambda_1']:.6f}")
        print(f"  Lambda_2: {spec['lambda_2']:.6f}")
        print(f"  Spectral gap: {spec['spectral_gap']:.6f}")
        print(f"  Mixing time estimate: {spec['mixing_time_est']:.1f} steps")
        print(f"  Influence concentration:")
        print(f"    - Top {spec['n_50_influence']} nodes ({100*spec['n_50_influence']/n:.1f}%) hold 50% influence")
        print(f"    - Top {spec['n_90_influence']} nodes ({100*spec['n_90_influence']/n:.1f}%) hold 90% influence")
    
    return results


# Run the analysis
spectral_results = run_spectral_analysis(networks)

In [ ]:
# Visualize spectral analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Eigenvalue spectrum comparison
ax = axes[0, 0]
for net_name, spec in spectral_results.items():
    eigenvalues = spec['eigenvalues'][:20]  # Top 20
    mags = [abs(e) for e in eigenvalues]
    ax.plot(range(len(mags)), mags, 'o-', label=net_name, markersize=4)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Eigenvalue Rank')
ax.set_ylabel('|Eigenvalue|')
ax.set_title('Eigenvalue Spectrum of Listening Matrix W')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)

# Plot 2: Spectral gap comparison
ax = axes[0, 1]
net_names = list(spectral_results.keys())
gaps = [spectral_results[n]['spectral_gap'] for n in net_names]
bars = ax.bar(range(len(net_names)), gaps, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(net_names)))
ax.set_xticklabels(net_names, rotation=45, ha='right')
ax.set_ylabel('Spectral Gap (1 - lambda_2)')
ax.set_title('Spectral Gap by Network\n(Larger = Faster Mixing)')
ax.grid(True, alpha=0.3, axis='y')
for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{gap:.3f}', ha='center', va='bottom', fontsize=8)

# Plot 3: Influence distribution (Lorenz-like curve)
ax = axes[1, 0]
for net_name, spec in spectral_results.items():
    stationary = spec['stationary_distribution']
    stationary_sorted = np.sort(stationary)[::-1]
    cumsum = np.cumsum(stationary_sorted) / np.sum(stationary_sorted)
    x = np.arange(1, len(stationary_sorted) + 1) / len(stationary_sorted)
    ax.plot(x, cumsum, '-', label=net_name, linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect equality')
ax.set_xlabel('Fraction of Nodes (ranked by influence)')
ax.set_ylabel('Cumulative Fraction of Influence')
ax.set_title('Influence Concentration (Lorenz Curve)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Summary table
ax = axes[1, 1]
ax.axis('off')
table_data = []
for net_name, spec in spectral_results.items():
    n = len(spec['nodes'])
    table_data.append([
        net_name[:16],
        f"{spec['spectral_gap']:.4f}",
        f"{spec['mixing_time_est']:.0f}",
        f"{spec['n_50_influence']} ({100*spec['n_50_influence']/n:.0f}%)",
        f"{spec['n_90_influence']} ({100*spec['n_90_influence']/n:.0f}%)",
    ])
table = ax.table(
    cellText=table_data,
    colLabels=['Network', 'Spectral Gap', 'Mix Time', 'N for 50%', 'N for 90%'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
ax.set_title('Spectral Properties Summary', fontsize=12, pad=20)

plt.tight_layout()
plt.savefig('spectral_analysis.png', dpi=150, bbox_inches='tight')
print("\nPlot saved to 'spectral_analysis.png'")
plt.show()

---
# Part 2: Root Node Influence Analysis

This section focuses on understanding how **root nodes** (agents with no incoming edges) determine
long-run outcomes in directed acyclic graphs (DAGs).

## Key Insight

In DAGs with root nodes:
- Root nodes act as **sources** of information
- Their final beliefs **propagate** to all their descendants
- The long-run outcome depends on which roots reach truth

## Long-Run Behavior Theorem (Informal)

**Claim:** In a DAG with root nodes, if we run the simulation long enough:
- All descendants of roots that converge to truth will converge to truth
- All descendants of roots that converge to falsehood will converge to falsehood
- The proportion reached by truthful roots predicts the final proportion believing truth

In [ ]:
# ============================================================================
# PART 2: ROOT NODE INFLUENCE ANALYSIS
# ============================================================================

def identify_roots(G):
    """
    Identify root nodes (in-degree = 0) in a directed graph.
    """
    roots = [n for n in G.nodes() if G.in_degree(n) == 0]
    return roots


def compute_reachability(G):
    """
    Compute reachability from each root to all other nodes.
    
    Returns:
        reach_matrix: (n_roots, n_nodes) boolean matrix
        roots: list of root node IDs
        nodes: list of all node IDs
    """
    roots = identify_roots(G)
    nodes = list(G.nodes())
    n = len(nodes)
    n_roots = len(roots)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    reach_matrix = np.zeros((n_roots, n), dtype=bool)
    
    for r_idx, root in enumerate(roots):
        # BFS/DFS to find all reachable nodes
        reachable = nx.descendants(G, root)
        reachable.add(root)  # Include the root itself
        
        for node in reachable:
            n_idx = node_to_idx[node]
            reach_matrix[r_idx, n_idx] = True
    
    return reach_matrix, roots, nodes


def predict_by_root_influence(G, root_beliefs, reach_matrix, roots, nodes):
    """
    Predict final beliefs based on root influence.
    
    For each node, predict truth if ANY reachable root believes truth.
    (This assumes truthful roots will eventually convert their descendants.)
    """
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    root_to_idx = {root: i for i, root in enumerate(roots)}
    
    predictions = np.zeros(n, dtype=bool)
    
    for node_idx, node in enumerate(nodes):
        # Check if any truthful root reaches this node
        truthful_root_reaches = False
        for r_idx, root in enumerate(roots):
            if reach_matrix[r_idx, node_idx] and root_beliefs[r_idx]:
                truthful_root_reaches = True
                break
        predictions[node_idx] = truthful_root_reaches
    
    return predictions


def compute_root_influence_metrics(G, root_beliefs, reach_matrix, roots, nodes):
    """
    Compute detailed metrics about root influence.
    """
    n = len(nodes)
    n_roots = len(roots)
    
    # Per-root metrics
    root_reach_counts = reach_matrix.sum(axis=1)  # How many nodes each root reaches
    root_reach_fractions = root_reach_counts / n
    
    # Truthful roots
    truthful_roots = np.array(root_beliefs)
    n_truthful_roots = truthful_roots.sum()
    
    # Nodes reached by at least one truthful root
    reached_by_truthful = np.any(reach_matrix[truthful_roots], axis=0)
    n_reached_by_truthful = reached_by_truthful.sum()
    frac_reached_by_truthful = n_reached_by_truthful / n
    
    return {
        'n_roots': n_roots,
        'n_truthful_roots': n_truthful_roots,
        'root_reach_counts': root_reach_counts,
        'root_reach_fractions': root_reach_fractions,
        'reached_by_truthful': reached_by_truthful,
        'n_reached_by_truthful': n_reached_by_truthful,
        'frac_reached_by_truthful': frac_reached_by_truthful,
    }

In [ ]:
def run_root_influence_analysis(G, network_name, n_experiments=10, uncertainty=0.001,
                                 n_steps=100000, n_trials=20):
    """
    Run simulations and analyze root influence prediction accuracy.
    """
    print(f"\n{'='*70}")
    print(f"ROOT INFLUENCE ANALYSIS: {network_name}")
    print(f"{'='*70}")
    
    nodes = list(G.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    # Compute reachability
    reach_matrix, roots, _ = compute_reachability(G)
    n_roots = len(roots)
    root_indices = [node_to_idx[r] for r in roots]
    
    print(f"\nNetwork: {n} nodes, {len(G.edges())} edges, {n_roots} roots")
    
    if n_roots == 0:
        print("  No root nodes found. Root influence analysis not applicable.")
        return None
    
    # Run trials
    all_results = []
    
    for trial in tqdm(range(n_trials), desc="Running trials"):
        # Run simulation with root analysis enabled
        model = VectorizedModel(
            network=G,
            n_experiments=n_experiments,
            uncertainty=uncertainty,
            agent_type="beta",
            tstep_stopping=True,
            compute_root_analysis=True,
        )
        model.run_simulation(number_of_steps=n_steps, show_bar=False)
        
        # Get actual outcomes
        actual_beliefs = model.credences[:, 1] > model.credences[:, 0]
        actual_proportion = np.mean(actual_beliefs)
        
        # Get root beliefs
        root_beliefs = actual_beliefs[root_indices]
        
        # Compute root influence metrics
        metrics = compute_root_influence_metrics(G, root_beliefs, reach_matrix, roots, nodes)
        
        # Predict based on root influence
        predictions = predict_by_root_influence(G, root_beliefs, reach_matrix, roots, nodes)
        predicted_proportion = np.mean(predictions)
        
        # Compare model's root analysis with ours
        model_prediction = model.proportion_reached_by_truth if model.root_analysis else None
        
        trial_results = {
            'actual_proportion': actual_proportion,
            'predicted_proportion': predicted_proportion,
            'model_prediction': model_prediction,
            'root_metrics': metrics,
            'prediction_error': abs(actual_proportion - predicted_proportion),
            'node_accuracy': np.mean(predictions == actual_beliefs),
        }
        all_results.append(trial_results)
    
    # Aggregate results
    actual_props = [r['actual_proportion'] for r in all_results]
    predicted_props = [r['predicted_proportion'] for r in all_results]
    errors = [r['prediction_error'] for r in all_results]
    node_accs = [r['node_accuracy'] for r in all_results]
    
    # Correlation between prediction and actual
    correlation = np.corrcoef(actual_props, predicted_props)[0, 1]
    
    summary = {
        'network': network_name,
        'n_nodes': n,
        'n_roots': n_roots,
        'n_trials': n_trials,
        'actual_mean': np.mean(actual_props),
        'actual_std': np.std(actual_props),
        'predicted_mean': np.mean(predicted_props),
        'predicted_std': np.std(predicted_props),
        'mean_error': np.mean(errors),
        'mean_node_accuracy': np.mean(node_accs),
        'correlation': correlation,
        'all_results': all_results,
    }
    
    print(f"\nResults across {n_trials} trials:")
    print(f"  Actual proportion: {summary['actual_mean']:.4f} +/- {summary['actual_std']:.4f}")
    print(f"  Root-predicted:    {summary['predicted_mean']:.4f} +/- {summary['predicted_std']:.4f}")
    print(f"  Mean error:        {summary['mean_error']:.4f}")
    print(f"  Node accuracy:     {summary['mean_node_accuracy']:.4f}")
    print(f"  Correlation:       {summary['correlation']:.4f}")
    
    return summary


# Run root influence analysis
print("\n" + "=" * 70)
print("ROOT NODE INFLUENCE ANALYSIS")
print("=" * 70)

root_analysis_results = {}

for net_name, net_info in networks.items():
    if net_info['has_roots']:
        G = net_info['graph']
        result = run_root_influence_analysis(
            G, net_name,
            n_experiments=10,
            uncertainty=0.001,
            n_steps=100000,
            n_trials=20
        )
        if result:
            root_analysis_results[net_name] = result

In [ ]:
# Visualize root influence analysis
if root_analysis_results:
    n_networks = len(root_analysis_results)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Predicted vs Actual scatter for all networks
    ax = axes[0, 0]
    colors = plt.cm.tab10(np.linspace(0, 1, n_networks))
    for idx, (net_name, result) in enumerate(root_analysis_results.items()):
        actual = [r['actual_proportion'] for r in result['all_results']]
        predicted = [r['predicted_proportion'] for r in result['all_results']]
        ax.scatter(predicted, actual, alpha=0.6, label=f"{net_name} (r={result['correlation']:.2f})",
                  color=colors[idx], s=50)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('Root-Predicted Proportion')
    ax.set_ylabel('Actual Proportion')
    ax.set_title('Root Influence Prediction Accuracy')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    # Plot 2: Prediction error comparison
    ax = axes[0, 1]
    net_names = list(root_analysis_results.keys())
    errors = [root_analysis_results[n]['mean_error'] for n in net_names]
    node_accs = [root_analysis_results[n]['mean_node_accuracy'] for n in net_names]
    
    x = np.arange(len(net_names))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, errors, width, label='Mean Error', color='coral', alpha=0.7)
    bars2 = ax.bar(x + width/2, node_accs, width, label='Node Accuracy', color='steelblue', alpha=0.7)
    
    ax.set_xticks(x)
    ax.set_xticklabels(net_names, rotation=45, ha='right')
    ax.set_ylabel('Value')
    ax.set_title('Root Prediction Performance')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Root reach distribution (first network with multiple roots)
    ax = axes[1, 0]
    plotted = False
    for net_name, result in root_analysis_results.items():
        if result['n_roots'] > 1:
            # Get reach fractions from first trial
            reach_fracs = result['all_results'][0]['root_metrics']['root_reach_fractions']
            ax.bar(range(len(reach_fracs)), sorted(reach_fracs, reverse=True), 
                   alpha=0.7, edgecolor='black')
            ax.set_xlabel('Root Rank (by reach)')
            ax.set_ylabel('Fraction of Network Reached')
            ax.set_title(f'Root Reach Distribution ({net_name})')
            ax.grid(True, alpha=0.3, axis='y')
            plotted = True
            break
    if not plotted:
        ax.text(0.5, 0.5, 'No multi-root network\nfor visualization', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Root Reach Distribution')
    
    # Plot 4: Summary table
    ax = axes[1, 1]
    ax.axis('off')
    table_data = []
    for net_name, result in root_analysis_results.items():
        table_data.append([
            net_name[:16],
            f"{result['n_roots']}",
            f"{result['mean_error']:.4f}",
            f"{result['mean_node_accuracy']:.4f}",
            f"{result['correlation']:.4f}",
        ])
    table = ax.table(
        cellText=table_data,
        colLabels=['Network', 'Roots', 'Mean Error', 'Node Acc', 'Correlation'],
        loc='center',
        cellLoc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    ax.set_title('Root Influence Summary', fontsize=12, pad=20)
    
    plt.tight_layout()
    plt.savefig('root_influence_analysis.png', dpi=150, bbox_inches='tight')
    print("\nPlot saved to 'root_influence_analysis.png'")
    plt.show()
else:
    print("No networks with root nodes found for analysis.")

## 2.2 Convergence of Root Prediction Over Time

An important question: How does the accuracy of root-based prediction improve as the simulation runs longer?

**Hypothesis:** As $t \to \infty$, the gap between root prediction and actual outcome should approach zero.

In [ ]:
def run_convergence_over_time(G, network_name, step_checkpoints, 
                              n_experiments=10, uncertainty=0.001, n_trials=10):
    """
    Track how root prediction accuracy improves over time.
    """
    print(f"\n{'='*60}")
    print(f"Convergence Analysis: {network_name}")
    print(f"Checkpoints: {step_checkpoints}")
    print(f"{'='*60}")
    
    nodes = list(G.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    reach_matrix, roots, _ = compute_reachability(G)
    root_indices = [node_to_idx[r] for r in roots]
    n_roots = len(roots)
    
    if n_roots == 0:
        print("  No roots found.")
        return None
    
    results_by_step = {step: [] for step in step_checkpoints}
    
    for trial in tqdm(range(n_trials), desc="Trials"):
        model = VectorizedModel(
            network=G,
            n_experiments=n_experiments,
            uncertainty=uncertainty,
            agent_type="beta",
            tstep_stopping=False,  # Don't stop early
            compute_root_analysis=True,
        )
        
        max_steps = max(step_checkpoints)
        current_step = 0
        
        for checkpoint in sorted(step_checkpoints):
            steps_to_run = checkpoint - current_step
            if steps_to_run > 0:
                model.run_simulation(number_of_steps=steps_to_run, show_bar=False)
                current_step = checkpoint
            
            # Compute metrics at this checkpoint
            actual_beliefs = model.credences[:, 1] > model.credences[:, 0]
            actual_proportion = np.mean(actual_beliefs)
            
            root_beliefs = actual_beliefs[root_indices]
            predictions = predict_by_root_influence(G, root_beliefs, reach_matrix, roots, nodes)
            predicted_proportion = np.mean(predictions)
            
            error = abs(actual_proportion - predicted_proportion)
            node_accuracy = np.mean(predictions == actual_beliefs)
            
            results_by_step[checkpoint].append({
                'actual': actual_proportion,
                'predicted': predicted_proportion,
                'error': error,
                'node_accuracy': node_accuracy,
            })
    
    # Aggregate
    summary = {
        'checkpoints': step_checkpoints,
        'mean_errors': [np.mean([r['error'] for r in results_by_step[s]]) for s in step_checkpoints],
        'std_errors': [np.std([r['error'] for r in results_by_step[s]]) for s in step_checkpoints],
        'mean_node_accs': [np.mean([r['node_accuracy'] for r in results_by_step[s]]) for s in step_checkpoints],
        'raw_results': results_by_step,
    }
    
    print(f"\nError by checkpoint:")
    for i, step in enumerate(step_checkpoints):
        print(f"  {step:>8} steps: error = {summary['mean_errors'][i]:.4f} +/- {summary['std_errors'][i]:.4f}")
    
    return summary


# Run convergence analysis
step_checkpoints = [1000, 5000, 10000, 25000, 50000, 100000]

convergence_results = {}
for net_name in ['barabasi_albert', 'tree']:
    if net_name in networks and networks[net_name]['has_roots']:
        G = networks[net_name]['graph']
        convergence_results[net_name] = run_convergence_over_time(
            G, net_name, step_checkpoints,
            n_experiments=10, uncertainty=0.001, n_trials=10
        )

In [ ]:
# Visualize convergence over time
if convergence_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Error over time
    ax = axes[0]
    for net_name, result in convergence_results.items():
        if result:
            ax.errorbar(result['checkpoints'], result['mean_errors'], 
                       yerr=result['std_errors'], marker='o', capsize=3,
                       label=net_name, linewidth=2, markersize=6)
    ax.set_xlabel('Simulation Steps')
    ax.set_ylabel('Prediction Error')
    ax.set_title('Root Prediction Error Over Time\n(Lower = Better)')
    ax.set_xscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Node accuracy over time
    ax = axes[1]
    for net_name, result in convergence_results.items():
        if result:
            ax.plot(result['checkpoints'], result['mean_node_accs'], 
                   marker='o', label=net_name, linewidth=2, markersize=6)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Perfect accuracy')
    ax.set_xlabel('Simulation Steps')
    ax.set_ylabel('Node-Level Accuracy')
    ax.set_title('Root Prediction Node Accuracy Over Time\n(Higher = Better)')
    ax.set_xscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.5, 1.05)
    
    plt.tight_layout()
    plt.savefig('convergence_over_time.png', dpi=150, bbox_inches='tight')
    print("\nPlot saved to 'convergence_over_time.png'")
    plt.show()

---
# Summary and Conclusions

In [ ]:
print("\n" + "=" * 80)
print("FORMAL ANALYSIS - FINAL SUMMARY")
print("=" * 80)

print("""
+------------------------------------------------------------------------------+
|                    STOCHASTIC PROCESS DECOMPOSITION                          |
+------------------------------------------------------------------------------+

The simulation can be understood through THREE key components:

1. STOCHASTIC MATRICES (Sampling Mechanism)
   - Binomial experiments with truth advantage 2u per trial
   - Evidence aggregates through network structure
   - Variance per step: n * p * (1-p) per source observed

2. NETWORK STRUCTURE (Information Flow)
   - The listening matrix W determines who influences whom
   - Spectral gap controls mixing time and convergence speed
   - Root nodes act as ultimate sources of information

3. INITIAL DISTRIBUTION (Random Priors)
   - Creates initial bias toward theories
   - Variance decomposition shows relative importance
   - High-influence agents' priors matter most

+------------------------------------------------------------------------------+
|                      ROOT NODE INFLUENCE ANALYSIS                            |
+------------------------------------------------------------------------------+

Key Findings:

* In DAGs with root nodes, the long-run outcome is determined by:
  1. Which roots converge to truth
  2. The reachability structure of the network

* Root-based prediction accuracy:
  - Improves with simulation length (as predicted by theory)
  - Node-level accuracy approaches 100% for long simulations
  - Error approaches 0 as t -> infinity

* Practical implications:
  - Root nodes are critical for network outcomes
  - Interventions targeting roots have maximum impact
  - Network design affects information flow patterns
""")

# Print specific results if available
if variance_results:
    print("\n" + "-" * 70)
    print("VARIANCE ANALYSIS RESULTS")
    print("-" * 70)
    for net_name, results in variance_results.items():
        print(f"  {net_name}:")
        print(f"    Proportion: {results['proportion_mean']:.4f} +/- {results['proportion_std']:.4f}")
        print(f"    Belief consistency: {results['belief_consistency'].mean():.4f}")

if init_analysis_results:
    print("\n" + "-" * 70)
    print("VARIANCE DECOMPOSITION RESULTS")
    print("-" * 70)
    for net_name, results in init_analysis_results.items():
        print(f"  {net_name}:")
        print(f"    Initial conditions: {results['frac_init']*100:.1f}% of variance")
        print(f"    Sampling randomness: {results['frac_sampling']*100:.1f}% of variance")

if spectral_results:
    print("\n" + "-" * 70)
    print("SPECTRAL ANALYSIS RESULTS")
    print("-" * 70)
    for net_name, spec in spectral_results.items():
        n = len(spec['nodes'])
        print(f"  {net_name}:")
        print(f"    Spectral gap: {spec['spectral_gap']:.4f}")
        print(f"    Est. mixing time: {spec['mixing_time_est']:.0f} steps")
        print(f"    Top {spec['n_50_influence']} nodes hold 50% influence")

if root_analysis_results:
    print("\n" + "-" * 70)
    print("ROOT INFLUENCE ANALYSIS RESULTS")
    print("-" * 70)
    for net_name, result in root_analysis_results.items():
        print(f"  {net_name} ({result['n_roots']} roots):")
        print(f"    Mean prediction error: {result['mean_error']:.4f}")
        print(f"    Node accuracy: {result['mean_node_accuracy']:.4f}")
        print(f"    Correlation: {result['correlation']:.4f}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
from datetime import datetime
try:
    import pytz
    nyc_time = datetime.now(pytz.timezone('America/New_York'))
    formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')
except ImportError:
    formatted_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f"Analysis completed at: {formatted_time}")

# Disconnect from Colab runtime to free resources
try:
    from IPython.display import Javascript
    display(Javascript('google.colab.kernel.disconnect()'))
    print("Disconnected from Colab runtime.")
except:
    print("(Not running in Colab - no disconnect needed)")